# Phase 1: AI Video Assembly Engine - Cloud Indexer
This notebook processes raw MP4 videos, standardizes them, extracts frames asynchronously, runs Microsoft Florence-2 for frame-by-frame description, and then utilizes Gemini 2.5 Flash to synthesize the results into a uniform JSON array for the local Assembly Engine.


## 1. Environment Setup & Dependencies


In [ ]:
!pip install -q transformers==4.41.2 timm accelerate einops opencv-python-headless google-generativeai ipywidgets nest_asyncio


## 2. Imports & Initialization


In [ ]:
import os
import cv2
import json
import time
import asyncio
import nest_asyncio
import subprocess
import shutil
from PIL import Image
import concurrent.futures
from IPython.display import display
import ipywidgets as widgets
import google.generativeai as genai
from google.colab import userdata
from transformers import AutoProcessor, AutoModelForCausalLM
import torch

# Allow asyncio to run in Jupyter
nest_asyncio.apply()

# Initialize Directories
RAW_DIR = "/content/raw_input"
NORMALIZED_DIR = "/content/normalized_input"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(NORMALIZED_DIR, exist_ok=True)

# Authenticate Gemini
print("Authenticating Gemini...")
gemini_key = userdata.get('GEMINI_API_KEY')
if not gemini_key:
    raise ValueError("Please add GEMINI_API_KEY to your Colab Secrets.")
genai.configure(api_key=gemini_key)
print("Initialization Complete. Please upload your `.mp4` files to `/content/raw_input/` before proceeding.")


## 3. Video Standardization Pipeline


In [ ]:
def standardize_videos(input_dir, output_dir):
    """Normalizes all .mp4 files to 720p 30fps using FFmpeg."""
    processed_files = []
    for file in os.listdir(input_dir):
        if not file.lower().endswith(".mp4"):
            continue
            
        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, f"std_{file}")
        
        print(f"Standardizing: {file}...")
        # FFmpeg command string
        cmd = [
            "ffmpeg", "-y", "-i", input_path,
            "-vf", "scale=1280:720",
            "-r", "30",
            "-c:v", "libx264",
            "-preset", "ultrafast",
            output_path
        ]
        
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if result.returncode == 0:
            print(f"Success -> {output_path}")
            processed_files.append(output_path)
        else:
            print(f"Failed to process {file}")
            
    return processed_files


## 4. Florence-2 Model Loading & Async Pipeline


In [ ]:
print("Loading Florence-2-large model (this may take a moment)...")
from transformers import AutoProcessor, AutoModelForCausalLM
from unittest.mock import patch
from transformers.dynamic_module_utils import get_imports

def mock_get_imports(filename: str | os.PathLike) -> list[str]:
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

with patch("transformers.dynamic_module_utils.get_imports", mock_get_imports):
    processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True).to("cuda").eval()

print("Model loaded on GPU.")

async def frame_producer(video_path, queue, fps=30):
    """Extracts 1 frame per second from the video and places it in an async queue."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video {video_path}")
        return

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / video_fps
    
    sec = 0
    while sec < duration:
        frame_id = int(sec * video_fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ret, frame = cap.read()
        
        if not ret:
            break
            
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb_frame)
        
        timestamp = time.strftime('%H:%M:%S', time.gmtime(sec))
        
        await queue.put((timestamp, pil_img))
        sec += 1
        await asyncio.sleep(0)
        
    cap.release()
    await queue.put(None)

def run_florence_batch(images):
    """Runs Florence inference synchronously."""
    task_prompt = "<DETAILED_CAPTION>"
    prompts = [task_prompt] * len(images)
    
    # We must run this on the main thread or properly handle cuda context if in threads,
    # but in Colab a thread pool usually handles model inference fine if not training.
    # Since model is already on CUDA:
    device = "cuda"
    inputs = processor(text=prompts, images=images, return_tensors="pt").to(device, torch.float16 if model.dtype == torch.float16 else torch.float32)
    
    with torch.no_grad():
        generated_ids = model.generate(
          input_ids=inputs["input_ids"],
          pixel_values=inputs["pixel_values"],
          max_new_tokens=1024,
          num_beams=3
        )
    
    generated_texts = processor.batch_decode(generated_ids, skip_special_tokens=False)
    results = []
    for text in generated_texts:
        parsed = processor.post_process_generation(text, task=task_prompt, image_size=(images[0].width, images[0].height))
        results.append(parsed[task_prompt])
        
    return results

async def frame_consumer(queue, results_dict, batch_size=4):
    """Pulls frames from the queue, batches them, and runs inference."""
    batch_images = []
    batch_timestamps = []
    
    loop = asyncio.get_running_loop()
    
    while True:
        item = await queue.get()
        if item is None:
            if batch_images:
                with concurrent.futures.ThreadPoolExecutor() as pool:
                    captions = await loop.run_in_executor(pool, run_florence_batch, batch_images)
                for ts, cap in zip(batch_timestamps, captions):
                    results_dict[ts] = cap
            queue.task_done()
            break
            
        ts, img = item
        batch_timestamps.append(ts)
        batch_images.append(img)
        
        if len(batch_images) >= batch_size:
            with concurrent.futures.ThreadPoolExecutor() as pool:
                captions = await loop.run_in_executor(pool, run_florence_batch, batch_images)
                
            for t_stamp, cap in zip(batch_timestamps, captions):
                results_dict[t_stamp] = cap
                
            batch_images.clear()
            batch_timestamps.clear()
            
        queue.task_done()

async def process_video_async(video_path):
    queue = asyncio.Queue(maxsize=10)
    results_dict = {}
    
    print(f"Extracting and analyzing frames for {os.path.basename(video_path)}...")
    producer_task = asyncio.create_task(frame_producer(video_path, queue))
    consumer_task = asyncio.create_task(frame_consumer(queue, results_dict))
    
    await asyncio.gather(producer_task, consumer_task)
    return results_dict


## 5. Gemini 2.5 Flash Scene Stitching


In [ ]:
def stitch_scenes_with_gemini(video_filename, frame_dict):
    """Calls Gemini API to group scenes based on second-by-second descriptions."""
    model = genai.GenerativeModel('gemini-2.5-flash')
    
    prompt = f"""
    Analyze this second-by-second visual description map of a video. 
    Group contiguous seconds where the visual scene, subject matter, and action remain contextually identical into unified clips. 
    For each grouped scene, provide a single cohesive summary paragraph (`clip_description`) and extract the single most dominant noun as the (`main_object`). 
    Output strictly as a valid JSON array matching this schema:
    [
      {{
        "video_source": "{video_filename}",
        "start_time": "00:00:12",
        "end_time": "00:00:18",
        "duration_seconds": 6.0,
        "main_object": "Noun",
        "clip_description": "A cohesive summary."
      }}
    ]
    
    Here is the mapping:
    {json.dumps(frame_dict, indent=2)}
    """
    
    response = model.generate_content(prompt, generation_config=genai.GenerationConfig(
        response_mime_type="application/json"
    ))
    
    try:
        return json.loads(response.text)
    except Exception as e:
        print(f"Error parsing Gemini response: {e}")
        return []


## 6. Main Orchestrator Execution


In [ ]:
async def main():
    final_results = []
    
    print("Step 1: Standardizing Videos...")
    std_videos = standardize_videos(RAW_DIR, NORMALIZED_DIR)
    
    if not std_videos:
        print("No videos found. Upload .mp4 files to /content/raw_input/ first.")
        return
        
    for v_path in std_videos:
        original_name = os.path.basename(v_path).replace("std_", "")
        
        print(f"\nStep 2: VLM Analysis for {original_name}...")
        frame_data = await process_video_async(v_path)
        
        print(f"Step 3: Stitching Scenes with Gemini...")
        stitched_data = stitch_scenes_with_gemini(original_name, frame_data)
        final_results.extend(stitched_data)
        
    print("\nSaving Final Results...")
    out_path = "/content/batch_results.json"
    with open(out_path, "w") as f:
        json.dump(final_results, f, indent=2)
    print(f"Pipeline Complete! Saved to {out_path}.")

# Run the pipeline
await main()


## 7. Interactive UI Purge Button


In [ ]:
def purge_raw_videos(b):
    try:
        shutil.rmtree(RAW_DIR)
        os.makedirs(RAW_DIR, exist_ok=True)
        print("Successfully purged heavy raw files from /content/raw_input/")
    except Exception as e:
        print(f"Error during purge: {e}")

button = widgets.Button(
    description='Purge Raw Videos',
    button_style='danger',
    tooltip='Click to wipe raw video files and free up space',
    icon='trash'
)

button.on_click(purge_raw_videos)
display(button)
